In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import random

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
torch.__version__

In [ ]:
torchvision.__version__

In [ ]:
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
print('number of train data :', len(train_df))
train_df.info()

In [ ]:
# Validate data value
print(train_df.isnull().values.any())
print(train_df.isnull().sum())
# Remove invalid data
#train_df = train_df.dropna(how='any')  # Remove rows that included Null value
#print(train_df.isnull().values.any())  # Check Null value

train_df['label'].value_counts().plot(kind='bar')

In [ ]:
indices_by_label = dict()

for index, label in train_df['label'].items():
    if False == (label in indices_by_label):
        indices_by_label[label] = list()
    
    indices_by_label[label].append(index)

num_classes = len(indices_by_label)
print(f'num_classes: {num_classes}')

In [ ]:
count_per_label = int(0)

for k, v in indices_by_label.items():
    if 0 >=  count_per_label:
        count_per_label = len(v)
    else:
        count_per_label = min(count_per_label, len(v))

    #print(f'label: {k}, total count: {len(v)}')

# The number of samples for all labels is adjusted to the minimum number of labels.
# Determine and apply the sample ratio of train and valid (train(8):valid(2)).
valid_data_count = count_per_label * 2 // 10
train_indices = []
valid_indices = []

for k, v in indices_by_label.items():
    raw_indices = torch.randperm(count_per_label).to(int)
    train_indices.extend(torch.Tensor(v)[raw_indices[valid_data_count:]].to(int))
    valid_indices.extend(torch.Tensor(v)[raw_indices[:valid_data_count]].to(int))

#print(f'train_indices length: {len(train_indices)}, {type(train_indices)}')
#print(f'valid_indices length: {len(valid_indices)}, {type(valid_indices)}')

#print(f'train_indices length: {len(torch.unique(torch.tensor(train_indices, dtype=int)))}')
#print(f'valid_indices length: {len(torch.unique(torch.tensor(valid_indices, dtype=int)))}')

# Check for duplicate indexes
assert len(train_indices) == len(torch.unique(torch.tensor(train_indices, dtype=int))), 'Duplicate train index exists.'
assert len(valid_indices) == len(torch.unique(torch.tensor(valid_indices, dtype=int))), 'Duplicate valid index exists.'

In [ ]:
class DigitDataset(torch.utils.data.Dataset):
    def __init__(self, raw_data: np.ndarray, indices: list):
        raw_tensors = torch.tensor(raw_data)
        self.labels = raw_tensors[indices, 0]
        self.inputs = raw_tensors[indices, 1:].reshape(-1, 1, 28, 28).repeat(1, 3, 1, 1).float() / 255.
    
    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        return self.inputs[index], self.labels[index]

    def __len__(self) -> int:
        return len(self.inputs)

In [ ]:
train_dataset = DigitDataset(train_df.values, train_indices)
valid_dataset = DigitDataset(train_df.values, valid_indices)

In [ ]:
np.set_printoptions(linewidth=400)

In [ ]:
rand_index = random.randrange(0, len(train_dataset))
rand_input, rand_label = train_dataset.__getitem__(rand_index)
print(f'label:{rand_label}')
rand_input[0].numpy()

In [ ]:
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')
print(f'device:{device}')

In [ ]:
learning_rate = 0.001
training_epochs = 20
batch_size = 128

In [ ]:
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class DigitRecognizer(nn.Module):
    def __init__(self, num_classes: int):
        super(DigitRecognizer, self).__init__()
        self.recognizer = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        fc_in_features = self.recognizer.fc.in_features
        self.recognizer.fc = nn.Linear(fc_in_features, num_classes)

    def forward(self, digit_image: torch.Tensor) -> torch.Tensor:
        return self.recognizer(digit_image)
    
    @torch.no_grad()
    def inference(self, digit_image: torch.Tensor) -> torch.Tensor:
        if 3 == len(digit_image.shape):
            digit_image = digit_image.unsqueeze(0)
        
        prediction = self.forward(digit_image)

        return torch.argmax(prediction, -1)

In [ ]:
model = DigitRecognizer(num_classes=num_classes)
model.to(device)

In [ ]:
criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
total_batch = len(train_dataloader)
print('number of total batch : {}'.format(total_batch))

In [ ]:
def calculate_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    # _, predicted = torch.max(logits, 1)
    predicted = torch.argmax(logits, dim=1)
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    accuracy = correct / total
    return accuracy

In [ ]:
def evaluate(model: nn.Module, valid_dataloader: torch.utils.data.Dataset, criterion: nn.Module | None, device: torch.device) -> tuple[float | None, float]:
    val_loss = 0
    val_correct = 0
    val_total = 0

    model.eval()
    with torch.no_grad():
        for images, labels in valid_dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)

            if criterion is not None:
                loss = criterion(logits, labels)
                val_loss += loss.item()
            
            val_correct += calculate_accuracy(logits, labels) * labels.size(0)
            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= len(valid_dataloader)

    return val_loss if criterion is not None else None, val_accuracy

In [ ]:
train_losses = []
valid_losses = []
best_valid_loss = float('inf')

for epoch in range(training_epochs):
    train_loss = 0
    train_correct = 0
    train_total = 0

    model.train()
    for inputs, labels in train_dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        with torch.autocast(device_type=str(device)):
            pred = model(inputs)
            cost = criterion(pred, labels)

        cost.backward()
        optimizer.step()
        optimizer.zero_grad()

        train_loss += cost.item()
        train_correct += calculate_accuracy(pred, labels) * labels.size(0)
        train_total += labels.size(0)

    train_accuracy = train_correct / train_total
    train_loss /= len(train_dataloader)
    train_losses.append(train_loss)

    # Validation
    valid_loss, valid_accuracy = evaluate(model, valid_dataloader, criterion, device)
    if valid_loss is not None:
        valid_losses.append(valid_loss)
    
    print(f'Epoch {epoch+1}/{training_epochs}:')
    print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')
    print(f'Validation Loss: {valid_loss:.4f}, Validation Accuracy: {valid_accuracy:.4f}')

    if valid_loss is not None and valid_loss < best_valid_loss:
        print(f'Validation loss improved from {best_valid_loss:.4f} to {valid_loss:.4f}.')
        print('Save checkpoint.')
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'model_best_checkpoint.pth')

In [ ]:
plt.plot(train_losses, color="blue", label='train loss')
plt.plot(valid_losses, color="red", label='valid loss')
plt.legend(loc='upper right')
plt.xticks(np.arange(0, len(train_losses), 2))
plt.xlabel('epochs')
plt.ylabel('loss')
plt.grid()
plt.show()

In [ ]:
model.load_state_dict(torch.load('model_best_checkpoint.pth'))
model.to(device)

In [ ]:
# Check saved model
valid_loss, valid_accuracy = evaluate(model, valid_dataloader, criterion, device)
print('Accuracy:', valid_accuracy)

In [ ]:
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')
test_df.info()

In [ ]:
test_inputs = torch.tensor(test_df.values).reshape(-1, 1, 28, 28).repeat(1, 3, 1, 1).to(device).float() / 255.
test_inputs.shape

In [ ]:
test_rand_index = int(np.random.uniform(test_inputs.shape[0]))
test_rand_input = test_inputs[test_rand_index]

model.eval()
test_predict = model.inference(test_rand_input)
test_predict

In [ ]:
test_rand_input[0].to('cpu').numpy()

In [ ]:
model.eval()
test_predicts = model.inference(test_inputs)
print(test_predicts.shape)

In [ ]:
num_check = 5
test_rand_index = int(np.random.uniform(test_inputs.shape[0] - num_check))
test_predicts[test_rand_index:test_rand_index+5]

In [ ]:
sample_submission_df = pd.read_csv('/kaggle/input/digit-recognizer/sample_submission.csv')
sample_submission_df.info()

In [ ]:
sample_submission_df.head()

In [ ]:
for index in range(len(sample_submission_df)):
    sample_submission_df.loc[index, 'Label'] = int(test_predicts[index])

In [ ]:
sample_submission_df[6234:6239]

In [ ]:
test_predicts[6234:6239]

In [ ]:
# Save submission file
#sample_submission_df.to_csv('/kaggle/input/digit-recognizer/sample_submission.csv', header=True, index=False)
#sample_submission_df.info()